# LeetCode 2219 — Maximum Sum Score of Array bằng Python

Link đề: https://leetcode.com/problems/maximum-sum-score-of-array/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum

> **Bài Premium:** đề trong notebook được viết lại theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, nên đối chiếu lại câu chữ và ví dụ với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def maximumSumScore(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Định nghĩa chính xác **tổng tiền tố** `nums[0..i]` (gồm `i + 1` phần tử) và **tổng hậu tố** `nums[i..n−1]` (gồm `n − i` phần tử) — chú ý **cả hai đều chứa `nums[i]`**.
2. Dùng mảng tiền tố `P` để tính mọi tổng đoạn trong `O(1)`: hậu tố từ `i` là `total − P[i]`.
3. Tối ưu xuống `O(1)` bộ nhớ bằng một biến tiền tố chạy và tổng `total`.
4. Nhận ra cấu trúc của đáp án: `max(max tiền tố khác rỗng, max hậu tố khác rỗng)`.
5. Kiểm thử với mảng **toàn âm** (đáp án âm) và đối chiếu brute force trên dữ liệu ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n`. **Điểm tổng** tại chỉ số `i` (`0 ≤ i < n`) là giá trị lớn hơn trong hai số:
- tổng `i + 1` phần tử **đầu** (`nums[0] + … + nums[i]`);
- tổng `n − i` phần tử **cuối** (`nums[i] + … + nums[n−1]`).

Trả về điểm tổng **lớn nhất** trên mọi chỉ số.

**Ví dụ 1:** `[4,3,-2,5]` → `10`. Tại `i = 0`: `max(4, 10) = 10`; `i = 1`: `max(7, 6) = 7`; `i = 2`: `max(5, 3) = 5`; `i = 3`: `max(10, 5) = 10`.

**Ví dụ 2:** `[-3,-5]` → `-3`. Tại `i = 0`: `max(-3, -8) = -3`; `i = 1`: `max(-8, -5) = -5`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `−10⁵ ≤ nums[i] ≤ 10⁵`.

In [1]:
import random
import time
from itertools import accumulate, product
from typing import List

examples = [([4, 3, -2, 5], 10), ([-3, -5], -3)]
for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp)

[4, 3, -2, 5] -> kỳ vọng 10
[-3, -5] -> kỳ vọng -3


## 3. Kiến thức nền

### 3.1 Mảng tiền tố

`P[0] = 0`, `P[k] = nums[0] + … + nums[k−1]` (độ dài `n + 1`). Tổng đoạn `nums[l..r]` (bao gồm) = `P[r+1] − P[l]`.

In [2]:
nums = [4, 3, -2, 5]
P = [0] + list(accumulate(nums))
print("P =", P, "| tổng nums[1..2] =", P[3] - P[1], "=", sum(nums[1:3]))

P = [0, 4, 7, 5, 10] | tổng nums[1..2] = 1 = 1


### 3.2 Tiền tố và hậu tố tại `i` (cả hai chứa `nums[i]`)

- tiền tố tại `i`: `nums[0..i]` = `P[i + 1]`;
- hậu tố tại `i`: `nums[i..n−1]` = `P[n] − P[i]` = `total − P[i]`.

In [3]:
n, total = len(nums), P[-1]
for i in range(n):
    print(f"i={i}: tiền tố nums[0..{i}] = {P[i + 1]:3d} | hậu tố nums[{i}..{n - 1}] = {total - P[i]:3d}")

i=0: tiền tố nums[0..0] =   4 | hậu tố nums[0..3] =  10
i=1: tiền tố nums[0..1] =   7 | hậu tố nums[1..3] =   6
i=2: tiền tố nums[0..2] =   5 | hậu tố nums[2..3] =   3
i=3: tiền tố nums[0..3] =  10 | hậu tố nums[3..3] =   5


### 3.3 `itertools.accumulate`

`accumulate(nums)` sinh các tổng tiền tố **khác rỗng** `nums[0], nums[0]+nums[1], …`. `accumulate(reversed(nums))` sinh các tổng hậu tố khác rỗng (từ cuối về đầu).

In [4]:
print(list(accumulate(nums)), list(accumulate(reversed(nums))))

[4, 7, 5, 10] [5, 3, 6, 10]


### 3.4 Khởi tạo khi đáp án có thể âm

Với mảng toàn âm, đáp án âm. Khởi tạo `best = float('-inf')` (hoặc bằng giá trị của chỉ số 0), **không** bằng `0`.

In [5]:
best = float("-inf")
for v in [-3, -8, -5]:
    best = max(best, v)
print(best)

-3


### 3.5 Đọc mảng tiền tố như "đồ thị độ cao"

Hình dung đi bộ từ trái sang phải, mỗi phần tử là một bước lên (dương) hoặc xuống (âm); `P[k]` là **độ cao** sau `k` bước. Khi đó:
- tổng tiền tố `nums[0..i]` = độ cao tại `i + 1` so với điểm xuất phát `P[0] = 0`;
- tổng hậu tố `nums[i..n−1]` = **độ cao cuối** `P[n]` trừ độ cao tại `i`.

Max tiền tố = **đỉnh cao nhất** (trừ điểm xuất phát); max hậu tố = độ cao cuối trừ **đáy thấp nhất** (trừ điểm cuối). Hình in bằng ký tự dưới đây cho ví dụ 1.

In [6]:
nums = [4, 3, -2, 5]
P = [0] + list(accumulate(nums))
lo = min(P)
for k, h in enumerate(P):
    print(f"P[{k}] = {h:3d} | " + " " * (h - lo) + "*")

P[0] =   0 | *
P[1] =   4 |     *
P[2] =   7 |        *
P[3] =   5 |      *
P[4] =  10 |           *


### 3.6 Slicing `nums[:i+1]` và `nums[i:]`

`nums[:i+1]` là `i + 1` phần tử đầu, `nums[i:]` là `n − i` phần tử cuối — **cả hai chứa `nums[i]`**. Mỗi slice tạo **bản sao** `O(độ dài)`; vì vậy `sum(nums[i:])` trong vòng lặp là `O(n²)` (Cách 1).

In [7]:
nums = [4, 3, -2, 5]; i = 2
print(nums[:i + 1], len(nums[:i + 1]), "|", nums[i:], len(nums[i:]), "| cùng chứa nums[2] =", nums[i])

[4, 3, -2] 3 | [-2, 5] 2 | cùng chứa nums[2] = -2


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1, lập bảng:

| i | tiền tố `nums[0..i]` | hậu tố `nums[i..3]` | điểm tổng |
|---|---|---|---|
| 0 | 4 | 10 | 10 |
| 1 | 7 | 6 | 7 |
| 2 | 5 | 3 | 5 |
| 3 | 10 | 5 | 10 |

Đáp án 10. Hai quan sát:
1. Tiền tố tại `i = n−1` và hậu tố tại `i = 0` đều là **tổng cả mảng** — nên đáp án luôn `≥ total`.
2. Lấy max trên mọi `i` của `max(tiền tố, hậu tố)` = `max(max mọi tiền tố, max mọi hậu tố)` — ta có thể tính hai max riêng.

Ví dụ 2 `[-3, -5]`: tiền tố `{-3, -8}`, hậu tố `{-8, -5}` ⇒ max `-3`.

In [8]:
for nums, _ in examples:
    pre = list(accumulate(nums)); suf = list(accumulate(reversed(nums)))[::-1]
    print(nums, "| tiền tố:", pre, "| hậu tố:", suf, "| điểm:", [max(a, b) for a, b in zip(pre, suf)])

[4, 3, -2, 5] | tiền tố: [4, 7, 5, 10] | hậu tố: [10, 6, 3, 5] | điểm: [10, 7, 5, 10]
[-3, -5] | tiền tố: [-3, -8] | hậu tố: [-8, -5] | điểm: [-3, -5]


**Khi nào đáp án lớn hơn hẳn tổng cả mảng?** Đáp án luôn `≥ total` (lấy tiền tố tại `i = n−1`). Nó lớn hơn `total` khi có một **tiền tố âm** ở đầu (bỏ nó đi thì hậu tố lớn hơn) hoặc một **hậu tố âm** ở cuối. Ví dụ tự dựng `[-4, 6, 2, -3]`: `total = 1`, nhưng hậu tố từ `i = 1` là `5` và tiền tố tới `i = 2` là `4` ⇒ đáp án `5`.

In [9]:
a = [-4, 6, 2, -3]
pre = list(accumulate(a)); suf = list(accumulate(reversed(a)))[::-1]
print("total =", sum(a), "| tiền tố:", pre, "| hậu tố:", suf, "| đáp án:", max(max(pre), max(suf)))

total = 1 | tiền tố: [-4, 2, 4, 1] | hậu tố: [1, 5, -1, -3] | đáp án: 5


## 5. Cách 1 — Brute Force / baseline: tính lại tổng cho từng `i`

**Ý tưởng:** đúng định nghĩa: `sum(nums[:i+1])`, `sum(nums[i:])`.

In [10]:
def score_brute(nums):
    n = len(nums)
    return max(max(sum(nums[:i + 1]), sum(nums[i:])) for i in range(n))

for nums, exp in examples:
    got = score_brute(nums); print(nums, "->", got); assert got == exp

[4, 3, -2, 5] -> 10
[-3, -5] -> -3


**Độ phức tạp:** mỗi `i` tốn `O(n)` ⇒ `O(n²)`; `n = 10⁵` ⇒ `10¹⁰`, TLE. Space `O(n)` (slice).

**Nút thắt:** tổng của `i + 1` phần tử đầu và của `i + 2` phần tử đầu chỉ khác nhau **một** phần tử — tính lại từ đầu là lãng phí.

In [11]:
rng = random.Random(0)
for n in (1000, 2000, 4000):
    a = [rng.randint(-10**5, 10**5) for _ in range(n)]
    t = time.perf_counter(); score_brute(a)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=1000: 0.006s
n=2000: 0.024s
n=4000: 0.097s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tiền tố):** với `P`, cả hai tổng tại `i` tính trong `O(1)`: `P[i+1]` và `total − P[i]`.

**Insight 2 (một lượt):** duyệt `i` tăng dần với biến `pre` = tổng `nums[0..i−1]` **trước** khi cộng `nums[i]`:
- hậu tố tại `i` = `total − pre` (tính **trước** khi cộng);
- cộng `nums[i]` vào `pre` ⇒ tiền tố tại `i` = `pre`.

**Insight 3 (tách hai max):** `max_i max(A_i, B_i) = max(max_i A_i, max_i B_i)`.

**Quy tắc "nếu… thì…":** mỗi bước **nếu** `total − pre` hoặc `pre + nums[i]` lớn hơn kỷ lục **thì** cập nhật.

**Invariant:** đầu vòng `i`, `pre = P[i]`.

**Khi nào tiền tố thắng, khi nào hậu tố thắng?** Tại `i`, `tiền tố − hậu tố = P[i+1] − (P[n] − P[i]) = 2·P[i] + nums[i] − total`. Như vậy với `i` nhỏ (khi `P[i]` còn nhỏ) hậu tố thường lớn hơn, với `i` lớn tiền tố thường lớn hơn — nhưng vì mảng có số âm nên không đơn điệu; ta cứ lấy max cả hai ở mọi `i`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng tiền tố `P`

In [12]:
def score_prefix(nums):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    total = P[n]
    return max(max(P[i + 1], total - P[i]) for i in range(n))

for nums, exp in examples:
    assert score_prefix(nums) == exp
print([score_prefix(nums) for nums, _ in examples])

[10, -3]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Một lượt, `O(1)` bộ nhớ (bản nộp chính)

In [13]:
def score_one_pass(nums):
    total = sum(nums)
    pre = 0
    best = float("-inf")
    for x in nums:
        best = max(best, total - pre)     # hậu tố bắt đầu tại x
        pre += x
        best = max(best, pre)             # tiền tố kết thúc tại x
    return best

for nums, exp in examples:
    assert score_one_pass(nums) == exp
print([score_one_pass(nums) for nums, _ in examples])

[10, -3]


**Độ phức tạp Cách 3:** `O(n)` (hai lượt: `sum` và vòng lặp), `O(1)` bộ nhớ.

### 7.3 Cách 4 — Hai max riêng bằng `accumulate`

In [14]:
def score_two_max(nums):
    return max(max(accumulate(nums)), max(accumulate(reversed(nums))))

for nums, exp in examples:
    assert score_two_max(nums) == exp
print([score_two_max(nums) for nums, _ in examples])

[10, -3]


**Độ phức tạp Cách 4:** `O(n)`, `O(1)` bộ nhớ thêm (generator). Ngắn gọn nhất, nhưng cần lập luận Insight 3.

### 7.4 Cách 5 — Hai con trỏ từ hai đầu (một vòng duy nhất)

Duyệt đồng thời tiền tố từ trái và hậu tố từ phải trong **một** vòng `for` (không cần `sum` trước).

In [15]:
def score_two_ends(nums):
    pre = suf = 0; best = float("-inf"); n = len(nums)
    for i in range(n):
        pre += nums[i]; suf += nums[n - 1 - i]
        best = max(best, pre, suf)
    return best

for nums, exp in examples:
    assert score_two_ends(nums) == exp
print([score_two_ends(nums) for nums, _ in examples])

[10, -3]


**Độ phức tạp Cách 5:** `O(n)`, `O(1)` bộ nhớ, đúng một lượt duyệt.

### 7.5 Cách 6 — Công thức "đỉnh và đáy" trên mảng tiền tố

Theo hình ảnh "độ cao" ở mục 3.5: max tiền tố khác rỗng = `max(P[1..n])`; max hậu tố khác rỗng = `P[n] − min(P[0..n−1])` (hậu tố bắt đầu tại `i` là `P[n] − P[i]`, lớn nhất khi `P[i]` nhỏ nhất). Vậy

$$\text{đáp án} = \max\big(\max_{1 \le k \le n} P[k],\ P[n] - \min_{0 \le k \le n-1} P[k]\big).$$

Công thức này cho thấy bài toán thực chất chỉ cần **ba** con số của mảng tiền tố: đỉnh, đáy (trừ điểm cuối) và độ cao cuối.

In [16]:
def score_peak_valley(nums):
    P = [0] + list(accumulate(nums))
    return max(max(P[1:]), P[-1] - min(P[:-1]))

for nums, exp in examples:
    assert score_peak_valley(nums) == exp
rng_ = random.Random(66)
for _ in range(500):
    a = [rng_.randint(-9, 9) for _ in range(rng_.randint(1, 12))]
    assert score_peak_valley(a) == score_brute(a)
print([score_peak_valley(nums) for nums, _ in examples], "| khớp brute force trên 500 mảng")

[10, -3] | khớp brute force trên 500 mảng


**Độ phức tạp Cách 6:** `O(n)` thời gian, `O(n)` bộ nhớ (mảng `P`; có thể làm một lượt `O(1)` bằng cách giữ đỉnh/đáy chạy).

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 (`total = 10`):

| i | x | hậu tố = total − pre (trước) | pre sau khi cộng (tiền tố) | best |
|---|---|---|---|---|
| 0 | 4 | 10 − 0 = 10 | 4 | 10 |
| 1 | 3 | 10 − 4 = 6 | 7 | 10 |
| 2 | −2 | 10 − 7 = 3 | 5 | 10 |
| 3 | 5 | 10 − 5 = 5 | 10 | 10 |

In [17]:
def trace_one_pass(nums):
    total = sum(nums); pre = 0; best = float("-inf")
    print("total =", total)
    for i, x in enumerate(nums):
        suf = total - pre; best = max(best, suf)
        pre += x; best = max(best, pre)
        print(f"i={i} x={x:3d}: hậu tố = {suf:4d} | tiền tố = {pre:4d} | điểm tại i = {max(suf, pre):4d} | best = {best}")
    return best

print("Đáp án:", trace_one_pass([4, 3, -2, 5]))
print()
print("Đáp án:", trace_one_pass([-3, -5]))

total = 10
i=0 x=  4: hậu tố =   10 | tiền tố =    4 | điểm tại i =   10 | best = 10
i=1 x=  3: hậu tố =    6 | tiền tố =    7 | điểm tại i =    7 | best = 10
i=2 x= -2: hậu tố =    3 | tiền tố =    5 | điểm tại i =    5 | best = 10
i=3 x=  5: hậu tố =    5 | tiền tố =   10 | điểm tại i =   10 | best = 10
Đáp án: 10

total = -8
i=0 x= -3: hậu tố =   -8 | tiền tố =   -3 | điểm tại i =   -3 | best = -3
i=1 x= -5: hậu tố =   -5 | tiền tố =   -8 | điểm tại i =   -5 | best = -3
Đáp án: -3


In [18]:
def trace_two_ends(nums):
    pre = suf = 0; best = float("-inf"); n = len(nums)
    for i in range(n):
        pre += nums[i]; suf += nums[n - 1 - i]; best = max(best, pre, suf)
        print(f"bước {i}: tiền tố nums[0..{i}] = {pre}, hậu tố nums[{n - 1 - i}..{n - 1}] = {suf}, best = {best}")
    return best

print("Đáp án:", trace_two_ends([2, -7, 3, -1, 4]))

bước 0: tiền tố nums[0..0] = 2, hậu tố nums[4..4] = 4, best = 4
bước 1: tiền tố nums[0..1] = -5, hậu tố nums[3..4] = 3, best = 4
bước 2: tiền tố nums[0..2] = -2, hậu tố nums[2..4] = 6, best = 6
bước 3: tiền tố nums[0..3] = -3, hậu tố nums[1..4] = -1, best = 6
bước 4: tiền tố nums[0..4] = 1, hậu tố nums[0..4] = 1, best = 6
Đáp án: 6


In [19]:
def trace_prefix_table(nums):
    n = len(nums); P = [0] + list(accumulate(nums)); total = P[n]
    print(f"P = {P}, total = {total}")
    print(" i | nums[i] | P[i+1] (tiền tố) | total-P[i] (hậu tố) | điểm")
    best = float("-inf")
    for i in range(n):
        a, b = P[i + 1], total - P[i]; best = max(best, a, b)
        print(f"{i:2d} | {nums[i]:7d} | {a:16d} | {b:19d} | {max(a, b):4d}")
    print("đáp án:", best)
    return best

trace_prefix_table([-4, 6, 2, -3])

P = [0, -4, 2, 4, 1], total = 1
 i | nums[i] | P[i+1] (tiền tố) | total-P[i] (hậu tố) | điểm
 0 |      -4 |               -4 |                   1 |    1
 1 |       6 |                2 |                   5 |    5
 2 |       2 |                4 |                  -1 |    4
 3 |      -3 |                1 |                  -3 |    1
đáp án: 5


5

**Đọc bảng:** cột "tiền tố" tăng/giảm theo dấu của `nums[i]`; cột "hậu tố" là "phản chiếu" của nó (`total − P[i]`). Điểm lớn nhất `5` đến từ **hậu tố** tại `i = 1` — bỏ đi phần đầu âm `−4`. Kiểm chứng thêm đẳng thức "max hậu tố = total − min tiền tố (trừ tiền tố đầy đủ)" trên dữ liệu ngẫu nhiên:

In [20]:
rng = random.Random(77)
for _ in range(500):
    a = [rng.randint(-9, 9) for _ in range(rng.randint(1, 12))]
    P = [0] + list(accumulate(a))
    assert max(accumulate(reversed(a))) == P[-1] - min(P[:-1])
    assert max(accumulate(a)) == max(P[1:])
print("max hậu tố = total − min(P[0..n−1]) và max tiền tố = max(P[1..n]) trên 500 mảng")

max hậu tố = total − min(P[0..n−1]) và max tiền tố = max(P[1..n]) trên 500 mảng


## 9. Code nộp LeetCode

In [21]:
class Solution:
    def maximumSumScore(self, nums: List[int]) -> int:
        total = sum(nums)
        pre = 0                       # tổng nums[0..i-1]
        best = float("-inf")
        for x in nums:
            best = max(best, total - pre)   # tổng n - i phần tử cuối (gồm x)
            pre += x
            best = max(best, pre)           # tổng i + 1 phần tử đầu (gồm x)
        return best

for nums, exp in examples:
    assert Solution().maximumSumScore(nums) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bất biến.** Đầu vòng lặp thứ `i`, `pre = P[i] = nums[0] + … + nums[i−1]`. Khi đó `total − pre = nums[i] + … + nums[n−1]` — đúng tổng `n − i` phần tử cuối. Sau khi `pre += nums[i]`, `pre = P[i+1]` — đúng tổng `i + 1` phần tử đầu. Vậy mỗi vòng lặp đưa **cả hai** đại lượng của chỉ số `i` vào `best`, và `best` = max trên mọi `i` của `max(tiền tố, hậu tố)` = đáp án.

**Tách hai max (Cách 4, 5).** Tập `{A_i} ∪ {B_i}` với `A_i` = tiền tố tại `i` (mọi tiền tố khác rỗng) và `B_i` = hậu tố tại `i` (mọi hậu tố khác rỗng). `max_i max(A_i, B_i)` = max của hợp hai tập = `max(max A, max B)`.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Tính lại tổng | `O(n²)` | `O(n)` |
| 2. Mảng tiền tố | `O(n)` | `O(n)` |
| 3. Một lượt | `O(n)` | `O(1)` |
| 4. Hai `accumulate` | `O(n)` | `O(1)` |
| 5. Hai đầu | `O(n)` | `O(1)` |
| 6. Đỉnh và đáy | `O(n)` | `O(n)` |

Tổng tuyệt đối tới `10⁵ · 10⁵ = 10¹⁰` — vượt `int32` (Java/C++ dùng `long`).

In [22]:
rng = random.Random(2219)
big = [rng.randint(-10**5, 10**5) for _ in range(10**5)]
neg = [-10**5] * 10**5
for name, fn in [("prefix", score_prefix), ("one_pass", score_one_pass), ("two_max", score_two_max),
                 ("two_ends", score_two_ends), ("Solution", lambda a: Solution().maximumSumScore(a))]:
    t = time.perf_counter(); r1 = fn(big); r2 = fn(neg)
    print(f"{name:9s}: ngẫu nhiên={r1}, toàn -1e5={r2} ({time.perf_counter() - t:.3f}s)")

prefix   : ngẫu nhiên=5767167, toàn -1e5=-100000 (0.026s)


one_pass : ngẫu nhiên=5767167, toàn -1e5=-100000 (0.015s)
two_max  : ngẫu nhiên=5767167, toàn -1e5=-100000 (0.008s)
two_ends : ngẫu nhiên=5767167, toàn -1e5=-100000 (0.017s)
Solution : ngẫu nhiên=5767167, toàn -1e5=-100000 (0.013s)


## 12. Bộ test

- Ví dụ đề.
- Biên: `n = 1` (cả hai tổng bằng `nums[0]`); toàn âm; toàn dương (đáp án là tổng); số 0; giá trị lớn nhất/nhỏ nhất.
- **Vét cạn** mọi mảng độ dài `1..6` trên `{−2..2}` và ngẫu nhiên lớn hơn (seed cố định).

In [23]:
fns = {"brute": score_brute, "prefix": score_prefix, "one_pass": score_one_pass, "two_max": score_two_max,
       "two_ends": score_two_ends, "peak_valley": score_peak_valley, "Solution": lambda a: Solution().maximumSumScore(a)}
edge = [([7], 7), ([-7], -7), ([0, 0, 0], 0), ([1, 2, 3], 6), ([-1, -2, -3], -1), ([-5, 10, -5], 5),
        ([10**5] * 3, 3 * 10**5), ([-3, 1, -3], -2)]
for nums, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(nums) == exp, (name, nums, fn(nums))
cnt = 0
for n in range(1, 7):
    for tup in product(range(-2, 3), repeat=n):
        a = list(tup); want = score_brute(a)
        for name, fn in fns.items():
            assert fn(a) == want, (name, a)
        cnt += 1
print(f"Ví dụ + biên + vét cạn {cnt} mảng: OK")

Ví dụ + biên + vét cạn 19530 mảng: OK


In [24]:
rng = random.Random(42)
for _ in range(300):
    a = [rng.randint(-10**5, 10**5) for _ in range(rng.randint(1, 200))]
    want = score_brute(a)
    for name, fn in fns.items():
        assert fn(a) == want, (name, a)
print("300 test ngẫu nhiên: OK")

300 test ngẫu nhiên: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Hậu tố lệch một: `total − P[i+1]` (bỏ mất `nums[i]`)

Đề nói "`n − i` phần tử cuối", tức **gồm** `nums[i]`.

In [25]:
def wrong_suffix_offset(nums):
    n = len(nums); P = [0] + list(accumulate(nums)); total = P[n]
    return max(max(P[i + 1], total - P[i + 1]) for i in range(n))        # SAI: hậu tố thiếu nums[i]

a = next(list(t) for n in range(1, 5) for t in product(range(-2, 3), repeat=n) if wrong_suffix_offset(list(t)) != score_brute(list(t)))
print("phản ví dụ ngắn nhất:", a, "| sai:", wrong_suffix_offset(a), "| đúng:", score_brute(a))

phản ví dụ ngắn nhất: [-2] | sai: 0 | đúng: -2


In [26]:
print("sửa (total - P[i]):", score_prefix(a))

sửa (total - P[i]): -2


### Lỗi 2 — Khởi tạo `best = 0`

In [27]:
def wrong_init_zero(nums):
    total = sum(nums); pre = 0; best = 0                                  # SAI
    for x in nums:
        best = max(best, total - pre); pre += x; best = max(best, pre)
    return best

print("sai:", wrong_init_zero([-3, -5]), "| đúng:", -3)

sai: 0 | đúng: -3


In [28]:
print("sửa:", Solution().maximumSumScore([-3, -5]))

sửa: -3


### Lỗi 3 — Tiền tố lệch một: dùng `P[i]` (chỉ `i` phần tử đầu)

Bản sai cho phép **tiền tố rỗng** (tổng 0) — với mảng toàn âm đáp án sai thành 0.

In [29]:
def wrong_prefix_offset(nums):
    n = len(nums); P = [0] + list(accumulate(nums)); total = P[n]
    return max(max(P[i], total - P[i]) for i in range(n))                # SAI: P[i] thay vì P[i+1]

print("sai:", wrong_prefix_offset([-3, -5]), "| đúng:", -3, "| ví dụ 1:", wrong_prefix_offset([4, 3, -2, 5]))

sai: 0 | đúng: -3 | ví dụ 1: 10


In [30]:
print("sửa:", score_prefix([-3, -5]), score_prefix([4, 3, -2, 5]))

sửa: -3 10


### Lỗi 4 — Chỉ xét tiền tố (quên hậu tố)

In [31]:
def wrong_prefix_only(nums):
    return max(accumulate(nums))                                          # SAI

a = [-3, 1, 2]
print("sai:", wrong_prefix_only(a), "| đúng:", score_brute(a))

sai: 0 | đúng: 3


In [32]:
print("sửa:", score_two_max(a))

sửa: 3


### Lỗi 5 — Cập nhật `pre` **trước** khi tính hậu tố trong Cách 3

Nếu cộng `x` vào `pre` trước, thì `total − pre` là hậu tố bắt đầu tại `i + 1` (thiếu `x`) — giống Lỗi 1.

In [33]:
def wrong_order(nums):
    total = sum(nums); pre = 0; best = float("-inf")
    for x in nums:
        pre += x                                                          # SAI: cộng trước
        best = max(best, total - pre, pre)
    return best

a = next(list(t) for n in range(1, 5) for t in product(range(-2, 3), repeat=n) if wrong_order(list(t)) != score_brute(list(t)))
print("phản ví dụ ngắn nhất:", a, "| sai:", wrong_order(a), "| đúng:", score_brute(a))

phản ví dụ ngắn nhất: [-2] | sai: 0 | đúng: -2


In [34]:
print("sửa:", Solution().maximumSumScore(a))

sửa: -2


### Lỗi 6 — Dùng Kadane (đoạn con **bất kỳ** có tổng lớn nhất)

Đề chỉ xét đoạn bắt đầu ở **đầu mảng** hoặc kết thúc ở **cuối mảng**. Kadane xét cả đoạn ở giữa ⇒ có thể cho kết quả **lớn hơn** đáp án đúng.

In [35]:
def wrong_kadane(nums):
    best = cur = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x); best = max(best, cur)
    return best                                               # SAI: đoạn ở giữa không được tính

a = [-1, 5, -1]
print("sai:", wrong_kadane(a), "| đúng:", score_brute(a), "(tiền tố/hậu tố tốt nhất đều là 4)")

sai: 5 | đúng: 4 (tiền tố/hậu tố tốt nhất đều là 4)


In [36]:
print("sửa:", Solution().maximumSumScore(a))

sửa: 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tổng `k` phần tử đầu / cuối"** ⇒ **tổng tiền tố**; hậu tố = `total − tiền tố`.
- Viết rõ mỗi tổng gồm **những phần tử nào** (bao gồm `nums[i]` hay không) trước khi code — lỗi lệch một rất phổ biến.
- `max` của các `max(A_i, B_i)` = `max(max A, max B)` — tách bài thành hai bài con đơn giản.
- Mảng có số âm ⇒ khởi tạo bằng `−∞` hoặc phần tử đầu.
- Tổng có thể vượt `int32`.

## 15. Biến thể của bài

1. **Điểm = `min` của hai tổng, tối đa** (bài tập 5) — chia mảng "cân bằng".
2. **LeetCode 2270 — Number of Ways to Split Array:** đếm `i` có tổng trái ≥ tổng phải (đoạn trái `0..i`, phải `i+1..n−1`).
3. **LeetCode 724 — Find Pivot Index:** tổng trái **bằng** tổng phải (không tính phần tử giữa).
4. **LeetCode 1991 — Find the Middle Index in Array:** như 724.
5. **Đoạn con tổng lớn nhất (Kadane, LeetCode 53):** tổng quát hơn — mọi đoạn, không chỉ tiền tố/hậu tố.

Minh họa biến thể 3 (Pivot Index, tự dựng, kiểm bằng brute force):

In [37]:
def pivot_index(nums):
    total = sum(nums); left = 0
    for i, x in enumerate(nums):
        if left == total - left - x:
            return i
        left += x
    return -1

def pivot_brute(nums):
    for i in range(len(nums)):
        if sum(nums[:i]) == sum(nums[i + 1:]): return i
    return -1

rng = random.Random(724)
for _ in range(500):
    a = [rng.randint(-3, 3) for _ in range(rng.randint(1, 10))]
    assert pivot_index(a) == pivot_brute(a)
print("Pivot index khớp brute force; [1,7,3,6,5,6] ->", pivot_index([1, 7, 3, 6, 5, 6]))

Pivot index khớp brute force; [1,7,3,6,5,6] -> 3


Minh họa biến thể 2 — **LeetCode 2270 (Number of Ways to Split Array)**: đếm chỉ số `i` (`0 ≤ i < n − 1`) sao cho tổng `nums[0..i]` ≥ tổng `nums[i+1..n−1]`. Khác bài gốc: hai phần **không** chung phần tử, và phần phải phải khác rỗng. Cùng một lượt tiền tố.

In [38]:
def ways_to_split(nums):
    total = sum(nums); pre = 0; cnt = 0
    for i in range(len(nums) - 1):
        pre += nums[i]
        cnt += pre >= total - pre
    return cnt

def ways_to_split_brute(nums):
    return sum(1 for i in range(len(nums) - 1) if sum(nums[:i + 1]) >= sum(nums[i + 1:]))

rng = random.Random(2270)
for _ in range(500):
    a = [rng.randint(-5, 5) for _ in range(rng.randint(2, 12))]
    assert ways_to_split(a) == ways_to_split_brute(a)
print("LeetCode 2270 khớp brute force; [10,4,-8,7] ->", ways_to_split([10, 4, -8, 7]))

LeetCode 2270 khớp brute force; [10,4,-8,7] -> 2


Minh họa biến thể 5 nâng cao — **LeetCode 918 (Maximum Sum Circular Subarray)**: mảng vòng, đoạn con có thể "quấn" qua cuối về đầu. Một đoạn quấn = **hậu tố + tiền tố** rời nhau = `total − (đoạn ở giữa bị bỏ)`. Nên đáp án = `max(Kadane max, total − Kadane min)` (trừ trường hợp mọi phần tử âm). Đây chính là mẹo "`total` trừ phần bị bỏ" của bài này, nâng lên một bậc. Kiểm bằng brute force mọi đoạn vòng.

In [39]:
def max_circular(nums):
    best_max = cur_max = nums[0]; best_min = cur_min = nums[0]
    for x in nums[1:]:
        cur_max = max(x, cur_max + x); best_max = max(best_max, cur_max)
        cur_min = min(x, cur_min + x); best_min = min(best_min, cur_min)
    total = sum(nums)
    return best_max if best_max < 0 else max(best_max, total - best_min)

def max_circular_brute(nums):
    n = len(nums); best = None
    for i in range(n):
        s = 0
        for L in range(1, n + 1):
            s += nums[(i + L - 1) % n]
            best = s if best is None or s > best else best
    return best

rng = random.Random(918)
for _ in range(500):
    a = [rng.randint(-6, 6) for _ in range(rng.randint(1, 9))]
    assert max_circular(a) == max_circular_brute(a)
print("LeetCode 918 khớp brute force; [5,-3,5] ->", max_circular([5, -3, 5]), "| [-3,-2,-3] ->", max_circular([-3, -2, -3]))

LeetCode 918 khớp brute force; [5,-3,5] -> 10 | [-3,-2,-3] -> -2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tính lại tổng | `sum(nums[:i+1])`, `sum(nums[i:])` | `O(n²)` | `O(n)` | kiểm tra |
| 2. Mảng tiền tố | `P[i+1]`, `total − P[i]` | `O(n)` | `O(n)` | rõ ràng, dễ mở rộng |
| 3. Một lượt | `pre` chạy + `total` | `O(n)` | `O(1)` | **nộp bài** |
| 4. Hai `accumulate` | max tiền tố, max hậu tố | `O(n)` | `O(1)` | ngắn nhất |
| 5. Hai đầu | tiền tố và hậu tố cùng vòng | `O(n)` | `O(1)` | một lượt, không cần `sum` |
| 6. Đỉnh và đáy | `max(max P[1:], P[n] − min P[:-1])` | `O(n)` | `O(n)` | hiểu bản chất hình học |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `maximumSumScore([1, -100, 1])`.
2. **Tính tay:** bảng tiền tố/hậu tố cho `[2, -7, 3, -1, 4]`.
3. **Điền TODO:** `sum_score_at(nums, i)` trả về điểm tổng tại chỉ số `i` trong `O(1)` khi đã có `P`.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `max_balanced_score(nums)` — điểm tại `i` là **`min`** (thay vì `max`) của hai tổng; trả về giá trị lớn nhất. Kiểm bằng brute force.

In [40]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def sum_score_at(P, i):
    # TODO: P là mảng tiền tố độ dài n + 1
    pass

In [41]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_maximum_sum_score(nums):
    # TODO
    pass

In [42]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def max_balanced_score(nums):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Tiền tố: `1, −99, −98`. (2) Hậu tố: `−98, −99, 1`. (3) Max `1`.

**Bài 2.** (1) `P = [0, 2, −5, −2, −3, 1]`, `total = 1`. (2) Tiền tố tại `i`: `2, −5, −2, −3, 1`; hậu tố: `1, −1, 6, 3, 4`. (3) Max `6` (hậu tố tại `i = 2`).

**Bài 3.** (1) `n = len(P) − 1`. (2) Tiền tố `P[i+1]`, hậu tố `P[n] − P[i]`. (3) `max` của hai số.

**Bài 4.** (1) `total = sum(nums)`. (2) Hậu tố trước, cộng `x`, tiền tố sau. (3) `best = −∞`.

**Bài 5.** (1) Thay `max` bằng `min` trong điểm tại `i`. (2) Không còn tách được thành hai max riêng. (3) Dùng Cách 2 hoặc 3 với `min`.

## 19. Đáp án tham khảo

In [43]:
# Bài 1, 2
assert Solution().maximumSumScore([1, -100, 1]) == 1
assert Solution().maximumSumScore([2, -7, 3, -1, 4]) == 6
print("Bài 1: 1 | Bài 2: 6")
trace_one_pass([2, -7, 3, -1, 4])

Bài 1: 1 | Bài 2: 6
total = 1
i=0 x=  2: hậu tố =    1 | tiền tố =    2 | điểm tại i =    2 | best = 2
i=1 x= -7: hậu tố =   -1 | tiền tố =   -5 | điểm tại i =   -1 | best = 2
i=2 x=  3: hậu tố =    6 | tiền tố =   -2 | điểm tại i =    6 | best = 6
i=3 x= -1: hậu tố =    3 | tiền tố =   -3 | điểm tại i =    3 | best = 6
i=4 x=  4: hậu tố =    4 | tiền tố =    1 | điểm tại i =    4 | best = 6


6

In [44]:
# Bài 3, 4
def sum_score_at_ans(P, i):
    n = len(P) - 1
    return max(P[i + 1], P[n] - P[i])

my_maximum_sum_score_ans = score_one_pass
rng = random.Random(3)
for _ in range(300):
    a = [rng.randint(-9, 9) for _ in range(rng.randint(1, 12))]
    P = [0] + list(accumulate(a))
    assert all(sum_score_at_ans(P, i) == max(sum(a[:i + 1]), sum(a[i:])) for i in range(len(a)))
    assert my_maximum_sum_score_ans(a) == score_brute(a)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [45]:
# Bài 5
def max_balanced_score_ans(nums):
    total = sum(nums); pre = 0; best = float("-inf")
    for x in nums:
        suf = total - pre; pre += x
        best = max(best, min(pre, suf))
    return best

def balanced_brute(nums):
    return max(min(sum(nums[:i + 1]), sum(nums[i:])) for i in range(len(nums)))

rng = random.Random(5)
for _ in range(500):
    a = [rng.randint(-9, 9) for _ in range(rng.randint(1, 12))]
    assert max_balanced_score_ans(a) == balanced_brute(a)
print("Bài 5: OK; ví dụ 1 ->", max_balanced_score_ans([4, 3, -2, 5]), "(so với 10 của bài gốc)")

Bài 5: OK; ví dụ 1 -> 6 (so với 10 của bài gốc)


**Giải thích thêm cho Bài 5.** Với `min` thay cho `max`, ta không tách được thành "hai max riêng" nữa: điểm tại `i` là `min(A_i, B_i)` với `A_i` tăng/giảm tùy dấu, `B_i = total − A_i + nums[i]`. Ta vẫn chỉ cần một lượt với `pre` chạy. Bảng dưới in `A_i`, `B_i`, `min` cho ví dụ 1 để thấy điểm cân bằng nhất nằm ở `i = 1` (tiền tố 7, hậu tố 6).

In [46]:
nums = [4, 3, -2, 5]; total = sum(nums); pre = 0
for i, x in enumerate(nums):
    suf = total - pre; pre += x
    print(f"i={i}: tiền tố {pre:3d}, hậu tố {suf:3d}, min = {min(pre, suf):3d}")

i=0: tiền tố   4, hậu tố  10, min =   4
i=1: tiền tố   7, hậu tố   6, min =   6
i=2: tiền tố   5, hậu tố   3, min =   3
i=3: tiền tố  10, hậu tố   5, min =   5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
total, pre, best = sum(nums), 0, float('-inf')
for x in nums:
    best = max(best, total - pre)   # hậu tố từ x
    pre += x
    best = max(best, pre)           # tiền tố tới x
return best
```

In [47]:
def template(nums):
    total, pre, best = sum(nums), 0, float("-inf")
    for x in nums:
        best = max(best, total - pre); pre += x; best = max(best, pre)
    return best
print([template(a) for a, _ in examples])

[10, -3]


**Checklist:**

- [ ] Tiền tố tại `i` gồm `nums[0..i]` (`i + 1` phần tử) — `P[i+1]`.
- [ ] Hậu tố tại `i` gồm `nums[i..n−1]` (`n − i` phần tử) — `total − P[i]`.
- [ ] Trong Cách 3: tính hậu tố **trước** khi cộng `x` vào `pre`.
- [ ] `best` khởi tạo `−∞` (mảng toàn âm).
- [ ] Tổng có thể vượt `int32`.
- [ ] Đã thử `n = 1`, toàn âm, ví dụ đề.